In [ ]:
# Imports and Configuration (hardwired)
import os
import math
import json
import csv
import tempfile
import numpy as np
from scipy.signal import medfilt
import librosa
import soundfile as sf
# Optional imports (MIDI / MusicXML)
try:
    from moviepy.editor import VideoFileClip
except Exception:
    VideoFileClip = None
try:
    import mido
except Exception:
    mido = None
try:
    import music21 as m21
except Exception:
    m21 = None
# Hardwired input/output paths and parameters
INPUT_FILE = r"C:\Users\hp\OneDrive\Documents\GitHub\somali-solfege-converter2\testVideo.mp4"
OUTPUT_PREFIX = r"C:\Users\hp\OneDrive\Documents\GitHub\somali-solfege-converter2\somali_detected"
TARGET_SR = 22050
FRAME_LENGTH = 2048
HOP_LENGTH = 128
YIN_THRESHOLD = 0.06
FREQ_MIN = 60
FREQ_MAX = 1200
SMOOTH_KERNEL = 3
MIN_NOTE_DURATION = 0.02
PITCH_TOLERANCE = 6
print('Configuration loaded. Input:', INPUT_FILE)

Configuration loaded. Input: C:\Users\hp\OneDrive\Documents\GitHub\somali-solfege-converter2\testVideo.mp4


## Audio / Signal Processing Functions
Helper functions for loading audio (video extraction if needed), pitch detection, smoothing, and segmentation.

In [10]:
def prepare_audio_input(path, target_sr=22050):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.mp4', '.mov', '.mkv') and VideoFileClip is not None:
        with VideoFileClip(path) as clip:
            tf = tempfile.NamedTemporaryFile(suffix='.wav', delete=False)
            tmp_path = tf.name
            tf.close()
            clip.audio.write_audiofile(tmp_path, verbose=False, logger=None)
            samples, sr = librosa.load(tmp_path, sr=target_sr, mono=True)
            try:
                os.unlink(tmp_path)
            except Exception:
                pass
    else:
        samples, sr = librosa.load(path, sr=target_sr, mono=True)
    return samples, sr


def yin_pitch_detection(samples, sr, frame_length=1024, hop_length=256, threshold=0.08, freq_min=60, freq_max=1000):
    # Prefer librosa.yin; returns (times, f0_list) with np.nan for unvoiced
    try:
        f0 = librosa.yin(samples, fmin=freq_min, fmax=freq_max, sr=sr, frame_length=frame_length, hop_length=hop_length, trough_threshold=threshold)
        times = librosa.frames_to_time(np.arange(len(f0)), sr=sr, hop_length=hop_length)
        return times.tolist(), f0.tolist()
    except Exception as e:
        raise


def smooth_pitch_track(pitches, kernel_size=7):
    arr = np.array(pitches, dtype=np.float32)
    nan_mask = np.isnan(arr)
    arr_filled = np.copy(arr)
    arr_filled[nan_mask] = 0.0
    if kernel_size % 2 == 0:
        kernel_size += 1
    sm = medfilt(arr_filled, kernel_size=kernel_size)
    sm[nan_mask] = np.nan
    return sm.tolist()


def hz_to_midi(freq):
    if freq is None or math.isnan(freq) or freq <= 0:
        return None
    return int(round(69 + 12 * math.log2(freq / 440.0)))


def segment_notes(times, pitches, min_note_duration=0.08, pitch_tolerance=12):
    """Segment voiced regions into notes and split on large pitch jumps.

    Parameters
    - times: frame times (list)
    - pitches: list of f0 (Hz) with np.nan for unvoiced
    - min_note_duration: minimum duration in seconds for a segment to be kept
    - pitch_tolerance: split if midi difference between consecutive voiced frames > this
    """
    notes = []
    n = len(pitches)
    voiced_mask = [not (p is None or math.isnan(p)) for p in pitches]
    start_idx = None
    for i, is_voiced in enumerate(voiced_mask):
        if is_voiced and start_idx is None:
            start_idx = i
        elif (not is_voiced or i == n - 1) and start_idx is not None:
            end_idx = i if not is_voiced else i + 1
            # indices of voiced frames inside region
            voiced_indices = [j for j in range(start_idx, end_idx) if voiced_mask[j]]
            if not voiced_indices:
                start_idx = None
                continue
            # iterate and split on pitch jumps
            sub_start = voiced_indices[0]
            prev_midi = hz_to_midi(pitches[sub_start])
            for idx in voiced_indices[1:]:
                cur_midi = hz_to_midi(pitches[idx])
                jump = None
                if prev_midi is not None and cur_midi is not None:
                    jump = abs(cur_midi - prev_midi)
                if jump is not None and jump > pitch_tolerance:
                    # end current subsegment at idx (exclusive)
                    s_t = times[sub_start]
                    e_t = times[idx - 1]
                    dur = e_t - s_t
                    if dur >= min_note_duration:
                        seg_pitches = [p for p in pitches[sub_start:idx] if not (p is None or math.isnan(p))]
                        if seg_pitches:
                            mean_pitch = float(np.mean(seg_pitches))
                            median_pitch = float(np.median(seg_pitches))
                            notes.append({
                                'start_time': float(s_t),
                                'end_time': float(e_t),
                                'duration': float(dur),
                                'mean_pitch': mean_pitch,
                                'median_pitch': median_pitch,
                                'midi_note': hz_to_midi(mean_pitch)
                            })
                    sub_start = idx
                    prev_midi = cur_midi
                else:
                    prev_midi = cur_midi
            # finalize final subsegment
            s_t = times[sub_start]
            e_t = times[voiced_indices[-1]]
            dur = e_t - s_t
            if dur >= min_note_duration:
                seg_pitches = [p for p in pitches[sub_start:voiced_indices[-1] + 1] if not (p is None or math.isnan(p))]
                if seg_pitches:
                    mean_pitch = float(np.mean(seg_pitches))
                    median_pitch = float(np.median(seg_pitches))
                    notes.append({
                        'start_time': float(s_t),
                        'end_time': float(e_t),
                        'duration': float(dur),
                        'mean_pitch': mean_pitch,
                        'median_pitch': median_pitch,
                        'midi_note': hz_to_midi(mean_pitch)
                    })
            start_idx = None
    return notes


## Solfege Conversion Logic
Map detected pitches to MIDI, note names, and (relative) solfege syllables. This notebook uses C major as the reference mapping (C=do, D=re, ...). Accidentals are annotated with `#` (sharp).

In [11]:
NOTE_NAMES = ['C','C#','D','D#','E','F','F#','G','G#','A','A#','B']
SOLFEGE_MAP = {'C':'do','D':'re','E':'mi','F':'fa','G':'so','A':'la','B':'ti'}

def midi_to_note_name(midi):
    if midi is None:
        return None
    pc = midi % 12
    octave = midi // 12 - 1
    return f"{NOTE_NAMES[pc]}{octave}"

def midi_to_solfege(midi):
    if midi is None:
        return None
    pc = midi % 12
    base = NOTE_NAMES[pc]  # e.g., 'C#' or 'D'
    if base.endswith('#'):
        natural = base[0]
        sol = SOLFEGE_MAP.get(natural, None)
        if sol is None:
            return base
        return sol + '#'
    else:
        return SOLFEGE_MAP.get(base, base)

def notes_to_solfege(notes):
    out = []
    for n in notes:
        midi = n.get('midi_note')
        name = midi_to_note_name(midi)
        sol = midi_to_solfege(midi)
        o = dict(n)
        o['note_name'] = name
        o['solfege'] = sol
        out.append(o)
    return out

## Execution / Workflow Orchestration
Run the full pipeline with the hardwired parameters, save JSON/CSV/MIDI/MusicXML (if available), and write a WAV preview.

In [12]:
# Run pipeline
samples, sr = prepare_audio_input(INPUT_FILE, target_sr=TARGET_SR)
print(f'Loaded {len(samples)/sr:.2f}s of audio at {sr}Hz')
times, raw_pitches = yin_pitch_detection(samples, sr, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH, threshold=YIN_THRESHOLD, freq_min=FREQ_MIN, freq_max=FREQ_MAX)
print('Detected frames:', len(times))
smoothed = smooth_pitch_track(raw_pitches, kernel_size=SMOOTH_KERNEL)
notes = segment_notes(times, smoothed, min_note_duration=MIN_NOTE_DURATION)
print('Segmented notes:', len(notes))
# Convert to solfege annotated notes
annotated = notes_to_solfege(notes)
# Save JSON and CSV
json_path = OUTPUT_PREFIX + '.json'
with open(json_path, 'w', encoding='utf-8') as jf:
    json.dump(annotated, jf, indent=2)
print('Saved JSON:', json_path)
csv_path = OUTPUT_PREFIX + '.csv'
fieldnames = ['start_time','end_time','duration','mean_pitch','median_pitch','midi_note','note_name','solfege']
with open(csv_path, 'w', newline='', encoding='utf-8') as cf:
    writer = csv.DictWriter(cf, fieldnames=fieldnames)
    writer.writeheader()
    for n in annotated:
        writer.writerow({k: n.get(k, '') for k in fieldnames})
print('Saved CSV:', csv_path)
# Write MIDI if available
if mido is not None and len(annotated) > 0:
    mid_path = OUTPUT_PREFIX + '.mid'
    mid = mido.MidiFile()
    track = mido.MidiTrack()
    mid.tracks.append(track)
    tempo_bpm = 120
    track.append(mido.MetaMessage('set_tempo', tempo=mido.bpm2tempo(tempo_bpm)))
    ticks_per_beat = mid.ticks_per_beat
    tempo = mido.bpm2tempo(tempo_bpm)
    ticks_per_second = ticks_per_beat * 1e6 / tempo
    last_tick = 0
    for n in annotated:
        start_tick = int(n['start_time'] * ticks_per_second)
        end_tick = int(n['end_time'] * ticks_per_second)
        delta = max(0, start_tick - last_tick)
        midi_note = n.get('midi_note')
        if midi_note is None:
            continue
        track.append(mido.Message('note_on', note=midi_note, velocity=64, time=delta))
        track.append(mido.Message('note_off', note=midi_note, velocity=64, time=(end_tick - start_tick)))
        last_tick = end_tick
    mid.save(mid_path)
    print('Saved MIDI:', mid_path)
else:
    print('Skipping MIDI export (mido missing or no notes)')
# Write MusicXML if music21 is available
if m21 is not None and len(annotated) > 0:
    s = m21.stream.Stream()
    s.append(m21.tempo.MetronomeMark(number=120))
    for n in annotated:
        midi = n.get('midi_note')
        if midi is None:
            continue
        dur_seconds = n.get('duration', 0.0)
        quarter_length = dur_seconds * (120.0 / 60.0)
        note_obj = m21.note.Note(m21.pitch.Pitch(midi=midi))
        note_obj.quarterLength = quarter_length
        s.append(note_obj)
    xml_path = OUTPUT_PREFIX + '.musicxml'
    s.write('musicxml', fp=xml_path)
    print('Saved MusicXML:', xml_path)
else:
    print('Skipping MusicXML export (music21 missing or no notes)')
# Synthesize a simple preview WAV (sine-based)
if len(annotated) > 0:
    total_dur = max(n['end_time'] for n in annotated)
    sr_preview = 44100
    out = np.zeros(int(total_dur * sr_preview) + 1, dtype=np.float32)
    for n in annotated:
        midi = n.get('midi_note')
        if midi is None:
            continue
        freq = 440.0 * (2.0 ** ((midi - 69) / 12.0))
        start = int(n['start_time'] * sr_preview)
        end = int(n['end_time'] * sr_preview)
        if end <= start:
            continue
        t = np.linspace(0, n['duration'], end-start, endpoint=False)
        tone = 0.12 * np.sin(2 * np.pi * freq * t)
        out[start:end] += tone
    maxv = np.max(np.abs(out))
    if maxv > 0:
        out = out / maxv * 0.9
    wav_path = OUTPUT_PREFIX + '_preview.wav'
    sf.write(wav_path, out, sr_preview)
    print('Saved preview WAV:', wav_path)
else:
    print('No notes to synthesize preview')

# Print a summary of first notes
print('\nFirst detected notes (up to 10):')
for n in annotated[:10]:
    print(n)

C:\Users\hp\AppData\Local\Temp\ipykernel_23928\1746246146.py:15: UserWarning: PySoundFile failed. Trying audioread instead.
  samples, sr = librosa.load(path, sr=target_sr, mono=True)


Loaded 73.54s of audio at 22050Hz
Detected frames: 12669
Segmented notes: 85
Saved JSON: C:\Users\hp\OneDrive\Documents\GitHub\somali-solfege-converter2\somali_detected.json
Saved CSV: C:\Users\hp\OneDrive\Documents\GitHub\somali-solfege-converter2\somali_detected.csv


PermissionError: [Errno 13] Permission denied: 'C:\\Users\\hp\\OneDrive\\Documents\\GitHub\\somali-solfege-converter2\\somali_detected.mid'

## Next Steps
- Run the notebook cells in order in a kernel that has the project `.venv` available.
- If `mido` or `music21` are missing, install them in the environment (`pip install mido music21`).
- Adjust `SMOOTH_KERNEL`, `MIN_NOTE_DURATION`, and `PITCH_TOLERANCE` to refine segmentation.